# Aviation Accident Safety — Data Cleaning

## Business problem

We are advising an aviation insurer that wants evidence-based guidance on aircraft safety. The client is interested in **professionally built airplanes that could plausibly still be active**, so the analysis is restricted to airplane accidents from **1983 onward**. The eventual analysis will compare small and large aircraft and evaluate two outcomes:

1. the fraction of occupants who were seriously or fatally injured; and
2. whether the aircraft was destroyed.

The goal of this notebook is to create a documented, reproducible cleaned dataset for the analysis notebook.

## Step 1 — Imports

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)

## Step 2 — Load and inspect the source data

The raw NTSB-style dataset contains aviation events from 1948–2023. Before altering it, we inspect dimensions, data types, missingness, and descriptive statistics.

In [ ]:
raw_df = pd.read_csv("data/AviationData.csv", encoding_errors="replace", low_memory=False)
print("Raw shape:", raw_df.shape)
raw_df.head()

In [ ]:
raw_df.info()

In [ ]:
missing_summary = (
    raw_df.isna().sum()
    .to_frame("missing")
    .assign(non_null=lambda x: len(raw_df) - x["missing"],
            pct_missing=lambda x: x["missing"] / len(raw_df))
    .sort_values("pct_missing", ascending=False)
)
missing_summary.head(15)

In [ ]:
raw_df.describe(include="all").T.head(20)

## Step 3 — Filter aircraft and events to the client's scope

### Assumptions
- `Aircraft.Category` must be **Airplane**.
- `Amateur.Built` must be **No**; unknown values are not assumed professional.
- The assignment specifies a 40-year retirement horizon relative to the 2023 dataset, so we use the fixed cutoff **1983-01-01**.
- Invalid dates are coerced to missing and therefore fail the date filter.

In [ ]:
df = raw_df.copy()
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")

print(raw_df["Aircraft.Category"].value_counts(dropna=False).head(10))
print(raw_df["Amateur.Built"].value_counts(dropna=False))
print("Date range:", df["Event.Date"].min(), "to", df["Event.Date"].max())

In [ ]:
df = df[
    df["Aircraft.Category"].astype("string").str.strip().str.lower().eq("airplane")
    & df["Amateur.Built"].astype("string").str.strip().str.lower().eq("no")
    & df["Event.Date"].ge(pd.Timestamp("1983-01-01"))
].copy()

print("Shape after client-scope filters:", df.shape)
print("Filtered date range:", df["Event.Date"].min(), "to", df["Event.Date"].max())

## Step 4 — Construct the injury and destruction metrics

### Injury metric
The four occupant-outcome columns (`Fatal`, `Serious`, `Minor`, `Uninjured`) are mutually exclusive outcome counts. Missing count cells are treated as **0**, a common encoding in this dataset when another injury category is populated. We then estimate the number of occupants as the sum of the four columns.

`Serious.Fatal.Fraction = (Fatal + Serious) / Total.Occupants`

Events with a computed occupant total of 0 are retained in the cleaned file, but their injury fraction is set to missing because a rate cannot be calculated.

### Aircraft destruction metric
`Destroyed` is encoded as 1 when `Aircraft.damage == "Destroyed"`, 0 for `Substantial` or `Minor`, and missing where damage is unknown. Unknown damage is **not** silently treated as a surviving aircraft.

In [ ]:
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
]

print(df[injury_cols].isna().sum())
df[injury_cols] = df[injury_cols].fillna(0)

df["Total.Occupants"] = df[injury_cols].sum(axis=1)
df["Serious.Fatal.Injuries"] = (
    df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]
)
df["Serious.Fatal.Fraction"] = np.where(
    df["Total.Occupants"] > 0,
    df["Serious.Fatal.Injuries"] / df["Total.Occupants"],
    np.nan,
)

df[["Total.Occupants", "Serious.Fatal.Fraction"]].describe()

In [ ]:
print(df["Aircraft.damage"].value_counts(dropna=False))

df["Aircraft.damage"] = (
    df["Aircraft.damage"].astype("string").str.strip().str.title()
    .replace({"Unknown": pd.NA, "Unk": pd.NA})
)

df["Destroyed"] = df["Aircraft.damage"].map(
    {"Destroyed": 1.0, "Substantial": 0.0, "Minor": 0.0}
)

df[["Aircraft.damage", "Destroyed"]].head()

## Step 5 — Clean and filter `Make`

The manufacturer field has substantial case and whitespace duplication (`CESSNA` vs `Cessna`, etc.) plus a handful of obvious corporate-name aliases. Cleaning steps:

1. trim whitespace;
2. normalize repeated whitespace;
3. uppercase labels;
4. collapse a small set of obvious manufacturer aliases;
5. retain only manufacturers represented by at least **50 events** after the client-scope filter.

The 50-event threshold prevents very sparse manufacturers from dominating later comparisons.

In [ ]:
make_aliases = {
    "CESSNA AIRCRAFT CO": "CESSNA",
    "CESSNA AIRCRAFT COMPANY": "CESSNA",
    "PIPER AIRCRAFT INC": "PIPER",
    "PIPER AIRCRAFT CORPORATION": "PIPER",
    "BEECHCRAFT": "BEECH",
    "BEECH AIRCRAFT": "BEECH",
    "BEECH AIRCRAFT CORP": "BEECH",
    "AIR TRACTOR INC": "AIR TRACTOR",
    "AIR TRACTOR INC.": "AIR TRACTOR",
    "CIRRUS DESIGN CORP": "CIRRUS",
    "CIRRUS DESIGN CORP.": "CIRRUS",
    "CIRRUS DESIGN CORPORATION": "CIRRUS",
    "DIAMOND AIRCRAFT IND INC": "DIAMOND",
    "DIAMOND AIRCRAFT INDUSTRIES": "DIAMOND",
    "MOONEY AIRCRAFT CORP": "MOONEY",
    "MOONEY AIRCRAFT CORP.": "MOONEY",
    "BOMBARDIER INC": "BOMBARDIER",
    "DEHAVILLAND": "DE HAVILLAND",
    "AIRBUS INDUSTRIE": "AIRBUS",
    "AVIAT AIRCRAFT INC": "AVIAT",
    "RAYTHEON AIRCRAFT COMPANY": "RAYTHEON",
    "ROCKWELL INTERNATIONAL": "ROCKWELL",
    "GRUMMAN AMERICAN AVN. CORP.": "GRUMMAN AMERICAN",
    "GRUMMAN AMERICAN AVN CORP": "GRUMMAN AMERICAN",
    "AMERICAN CHAMPION AIRCRAFT": "AMERICAN CHAMPION",
}

def clean_make(value):
    if pd.isna(value):
        return np.nan
    label = re.sub(r"\s+", " ", str(value).strip().upper())
    return make_aliases.get(label, label)

before = df["Make"].value_counts().head(15)
df["Make"] = df["Make"].map(clean_make)
df = df.dropna(subset=["Make"])
after = df["Make"].value_counts().head(15)

print("Before normalization:")
print(before)
print()
print("After normalization:")
print(after)

In [ ]:
make_counts = df["Make"].value_counts()
eligible_makes = make_counts[make_counts >= 50].index
df = df[df["Make"].isin(eligible_makes)].copy()

print("Retained makes:", df["Make"].nunique())
print("Rows after make-frequency filter:", len(df))
df["Make"].value_counts().head(15)

## Step 6 — Clean `Model` and create a unique airplane-type key

Model labels are not globally unique across manufacturers. We drop missing/blank models, normalize whitespace and case, then combine manufacturer and model into `Plane.Type`.

In [ ]:
df = df.dropna(subset=["Model"]).copy()
df["Model"] = (
    df["Model"].astype(str).str.strip().str.upper()
    .str.replace(r"\s+", " ", regex=True)
)
df = df[df["Model"].ne("")].copy()
df["Plane.Type"] = df["Make"] + " " + df["Model"]

print("Unique raw model labels:", df["Model"].nunique())
print("Unique make/model plane types:", df["Plane.Type"].nunique())
df[["Make", "Model", "Plane.Type"]].head()

## Step 7 — Clean other explanatory variables

We preserve missingness rather than inventing values. The following standardization is applied:

- `Engine.Type`: trim/title-case; Unknown/UNK/NONE-like placeholders → missing.
- `Weather.Condition`: trim/uppercase; unknown placeholders → missing.
- `Number.of.Engines`: coerce numeric; zero/negative values → missing because these are airplanes.
- `Purpose.of.flight`: trim; unknown placeholders → missing.
- `Broad.phase.of.flight`: trim/title-case; unknown placeholders → missing.

Rare categories are not removed here; sample-size thresholds are applied during analysis instead.

In [ ]:
df["Engine.Type"] = (
    df["Engine.Type"].astype("string").str.strip().str.title()
    .replace({"Unknown": pd.NA, "Unk": pd.NA, "None": pd.NA, "Lr": pd.NA})
)

df["Weather.Condition"] = (
    df["Weather.Condition"].astype("string").str.strip().str.upper()
    .replace({"UNK": pd.NA, "UNKNOWN": pd.NA})
)

df["Number.of.Engines"] = pd.to_numeric(df["Number.of.Engines"], errors="coerce")
df.loc[df["Number.of.Engines"] <= 0, "Number.of.Engines"] = np.nan

df["Purpose.of.flight"] = (
    df["Purpose.of.flight"].astype("string").str.strip()
    .replace({"Unknown": pd.NA, "UNKNOWN": pd.NA, "Unk": pd.NA})
)

df["Broad.phase.of.flight"] = (
    df["Broad.phase.of.flight"].astype("string").str.strip().str.title()
    .replace({"Unknown": pd.NA, "Unk": pd.NA})
)

for col in ["Engine.Type", "Weather.Condition", "Number.of.Engines", "Purpose.of.flight", "Broad.phase.of.flight"]:
    print()
    print(col)
    print(df[col].value_counts(dropna=False).head(12))

## Step 8 — Remove very sparse source columns

The assignment asks us to retain source columns with more than **20,000 non-null observations**. We calculate that threshold on the original 88,889-row source table, before client filtering; otherwise the post-filtered table itself is smaller than 20,000 rows and useful required fields such as weather and engine type would be removed.

Derived columns created in this notebook are retained regardless of the source threshold.

In [ ]:
raw_non_null = raw_df.notna().sum()
source_keep = raw_non_null[raw_non_null > 20_000].index.tolist()
dropped_source_columns = [c for c in raw_df.columns if c not in source_keep]
print("Source columns removed for low completeness:", dropped_source_columns)

derived_columns = [
    "Total.Occupants",
    "Serious.Fatal.Injuries",
    "Serious.Fatal.Fraction",
    "Destroyed",
    "Plane.Type",
]

keep_columns = list(dict.fromkeys([c for c in source_keep if c in df.columns] + derived_columns))
df_clean = df[keep_columns].copy()
print("Cleaned shape:", df_clean.shape)
df_clean.info()

## Step 9 — Save the cleaned dataset

The analysis notebook loads this intermediate file directly, keeping the workflow modular and reproducible.

In [ ]:
output_path = "data/AviationData_cleaned.csv"
df_clean.to_csv(output_path, index=False)
print(f"Saved {len(df_clean):,} rows and {df_clean.shape[1]} columns to {output_path}")

## Cleaning summary

The cleaned dataset is restricted to professionally built airplanes from 1983 onward, contains only manufacturers with at least 50 scoped events, has standardized make/model labels and a unique `Plane.Type`, and includes two explicit safety outcomes. Missing or unknown damage and explanatory-factor values remain missing rather than being converted into favorable outcomes.